In [2]:
import hashlib
import json
import pickle
import subprocess
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import URLError, HTTPError

Setup

In [6]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys

def locate_reference():
    here = Path.cwd().resolve()
    candidates = []
    for folder in [here, *here.parents]:
        candidates += [folder / "reference", folder]
        try:
            candidates += [p for p in folder.iterdir() if p.is_dir() and p.name.lower() == "reference"]
        except (OSError, PermissionError):
            pass
    seen = set()
    for p in candidates:
        p = p.resolve()
        if p not in seen and (p / "pyproject.toml").is_file() and (p / "uv.lock").is_file():
            return p
        seen.add(p)
    return None

REFERENCE_DIR = locate_reference()
UV = shutil.which("uv")
print("Notebook folder:", Path.cwd())
print("Reference folder:", REFERENCE_DIR)
print("uv executable:", UV)
if REFERENCE_DIR is None:
    print(r'If needed, set REFERENCE_DIR = Path(r"C:\full\path\to\reference")')
if UV is None:
    print("Install uv and restart VS Code before continuing.")

Notebook folder: c:\Users\ASUS\OneDrive\Desktop\machine-learning-zoomcamp-homework\05-Deploying Machine Learning Models
Reference folder: C:\Users\ASUS\OneDrive\Desktop\machine-learning-zoomcamp-homework\05-Deploying Machine Learning Models\reference
uv executable: C:\Users\ASUS\.local\bin\uv.EXE


In [7]:
if REFERENCE_DIR is None:
    raise FileNotFoundError("Could not find reference folder. Set REFERENCE_DIR manually in the previous cell.")
if UV is None:
    raise FileNotFoundError("uv was not found on PATH. Install uv, restart VS Code, and rerun.")

for command in ([UV, "sync", "--locked"], [UV, "run", "python", "smoke_test.py"]):
    result = subprocess.run(command, cwd=REFERENCE_DIR, text=True, capture_output=True)
    print("$", " ".join(command))
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode:
        raise RuntimeError(f"Command failed: {' '.join(command)}")

$ C:\Users\ASUS\.local\bin\uv.EXE sync --locked

 Downloaded cpython-3.11.15-windows-x86_64-none (download)
Using CPython 3.11.15
Creating virtual environment at: .venv
Resolved 19 packages in 1ms
 Downloaded pydantic-core
 Downloaded scikit-learn
 Downloaded numpy
 Downloaded scipy
Prepared 18 packages in 2m 23s
Installed 18 packages in 1.92s
 + annotated-types==0.8.0
 + anyio==4.15.1
 + click==8.5.0
 + cloudpickle==3.1.2
 + fastapi==0.119.1
 + h11==0.16.0
 + idna==3.19
 + joblib==1.6.0
 + numpy==2.3.3
 + pydantic==2.13.5
 + pydantic-core==2.46.5
 + scikit-learn==1.7.2
 + scipy==1.17.1
 + starlette==0.48.0
 + threadpoolctl==3.6.0
 + typing-extensions==4.16.0
 + typing-inspection==0.4.4
 + uvicorn==0.38.0

$ C:\Users\ASUS\.local\bin\uv.EXE run python smoke_test.py
{'conversion_probability': 0.769799, 'conversion': True}



In [8]:
expected_checksum = "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1"
actual_checksum = hashlib.sha256((REFERENCE_DIR / "pipeline.bin").read_bytes()).hexdigest()
print("Actual:  ", actual_checksum)
print("Expected:", expected_checksum)
assert actual_checksum == expected_checksum, "Model checksum mismatch"
print("Checksum verified.")

Actual:   1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1
Expected: 1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1
Checksum verified.


Question 1: environment check

In [9]:
r = subprocess.run([UV, "--version"], cwd=REFERENCE_DIR, capture_output=True, text=True, check=True)
print(r.stdout.strip())

uv 0.13.5 (46776a95d 2026-10-21 x86_64-pc-windows-msvc)


Question 2 : locked dependency

In [10]:
import tomllib

with (REFERENCE_DIR / "uv.lock").open("rb") as f:
    lock_data = tomllib.load(f)
versions = [p["version"] for p in lock_data["package"] if p["name"] == "scikit-learn"]
print("Scikit-Learn version in uv.lock:", versions)
assert versions == ["1.7.2"]

Scikit-Learn version in uv.lock: ['1.7.2']


Question 3 : load the model

In [12]:
q3_code = '''
import pickle
from model import normalize_record
lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}
with open("pipeline.bin", "rb") as f:
    pipeline = pickle.load(f)
record = normalize_record(lead)
probability = float(pipeline.predict_proba([record])[0, 1])
print(f"Unrounded probability: {probability:.10f}")
print(f"Q3 answer: {probability:.3f}")
'''
r = subprocess.run([UV, "run", "python", "-c", q3_code], cwd=REFERENCE_DIR, capture_output=True, text=True)
print(r.stdout)
if r.stderr: print(r.stderr)
if r.returncode: raise RuntimeError("Q3 prediction failed")

Unrounded probability: 0.5329944186
Q3 answer: 0.533



Question 4 : serve the model

In [18]:
from urllib.request import Request, urlopen
from urllib.error import URLError, HTTPError

API_URL = "http://127.0.0.1:9696"
lead_q4 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}
try:
    with urlopen(f"{API_URL}/health", timeout=5) as response:
        print("Health:", json.load(response))
    req = Request(f"{API_URL}/predict", data=json.dumps(lead_q4).encode(),
                  headers={"Content-Type": "application/json"}, method="POST")
    with urlopen(req, timeout=10) as response:
        prediction = json.load(response)
    print("Prediction:", prediction)
    probability = float(prediction["conversion_probability"])
    assert 0 <= probability <= 1
    assert isinstance(prediction["conversion"], bool)
    print(f"Q4 answer: {probability:.3f}")
except (URLError, HTTPError, TimeoutError) as e:
    print("API unavailable. Start uvicorn in another terminal, then rerun this cell.")
    print("Details:", e)

Health: {'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}
Prediction: {'conversion_probability': 0.769799, 'conversion': True}
Q4 answer: 0.770


Question 5 :  look at the container configuration


In [19]:
dockerfile = (REFERENCE_DIR / "Dockerfile").read_text(encoding="utf-8")
from_lines = [line.strip() for line in dockerfile.splitlines() if line.strip().upper().startswith("FROM ")]
print("\n".join(from_lines))
assert any("python:3.11.15-slim-bookworm" in line for line in from_lines)
print("Q5 answer: python:3.11.15-slim-bookworm")

FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3
Q5 answer: python:3.11.15-slim-bookworm


Question 6 :  run the container

In [20]:
# Optional: runs q6_test.py against whichever service is listening on localhost:9696.
# For the homework's Q6, make sure that service is the Docker container.
r = subprocess.run([UV, "run", "python", "q6_test.py"], cwd=REFERENCE_DIR,
                   capture_output=True, text=True)
print(r.stdout)
if r.stderr: print(r.stderr)
if r.returncode:
    print("Check that Docker is running, the container is up, and port 9696 is available.")

{'conversion_probability': 0.769799, 'conversion': True}

